In [0]:
# Databricks notebook source
# 06_ads_autoloader

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

# =========================================================
# CONFIG
# =========================================================
LANDING_PATH = "/Volumes/workspace/00_landing/landing/api_ads_raw"
CHECKPOINT_PATH = "/Volumes/workspace/00_landing/landing/checkpoints/ads"

# sửa lại nếu bronze table của bạn nằm chỗ khác
BRONZE_ADS_TABLE = "workspace.01_bronze.bronze_ads_campaigns"
# ví dụ:
# BRONZE_ADS_TABLE = "workspace.`01_bronze`.ads_campaigns"

schema = StructType([
    StructField("date", StringType()),
    StructField("campaign_id", StringType()),
    StructField("campaign_name", StringType()),
    StructField("channel", StringType()),
    StructField("source", StringType()),
    StructField("medium", StringType()),
    StructField("impressions", IntegerType()),
    StructField("clicks", IntegerType()),
    StructField("conversions", IntegerType()),
    StructField("cost", DoubleType()),
    StructField("currency", StringType())
])

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .schema(schema)
    .load(LANDING_PATH)
)

stream_clean = (
    stream_df
    .withColumn("date", F.to_date("date"))
    .withColumn("ingestion_time", F.current_timestamp())
    .withColumn("source_api", F.lit("mock_olist_ads"))
)

def upsert_ads(microBatchDF, batchId):
    microBatchDF = (
        microBatchDF
        .filter(F.col("date").isNotNull() & F.col("campaign_id").isNotNull())
        .dropDuplicates(["date", "campaign_id"])
    )

    delta_table = DeltaTable.forName(spark, BRONZE_ADS_TABLE)

    (
        delta_table.alias("t")
        .merge(
            microBatchDF.alias("s"),
            "t.date = s.date AND t.campaign_id = s.campaign_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    stream_clean.writeStream
    .foreachBatch(upsert_ads)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start()
)

query.awaitTermination()
print("ads autoloader completed")